# Validation Test Evaluation - Droplet in Equilibrium (transient simulations)

This notebook and the corresponding simulation setup notebook (DropletInEquilibrium_transient_Run.ipynb) are part of published results (cf. 7.2.1) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
wmg.Init("XNSEpaper_Droplet");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\XNSEpaper_Droplet");

Opening existing database '\\fdygitrunner\ValidationTests\databases\XNSEpaper_Droplet'.


In [3]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
var sessions = wmg.Sessions;
//var sessions = databases.Pick(0).Sessions;
sessions

#0: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh80	11/11/2025 2:07:51 PM	7fd6725f...
#1: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh60	11/11/2025 2:07:41 PM	e0bdcb6c...
#2: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh40	11/11/2025 2:07:31 PM	1f1fc172...
#3: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh20	11/11/2025 2:07:20 PM	4b87a5cb...
#4: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh10	11/11/2025 2:07:10 PM	7c499446...
#5: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh40	10/24/2025 4:48:06 PM	2791d068...
#6: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh20	10/24/2025 4:47:55 PM	ef32ca7d...


In [5]:
int[] Resolutions = { 20, 40, 60, 80 };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct14_000000.XNSEpaper_Droplet");

In [7]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
foreach (int Res in Resolutions) {
    string studyName = $"DropletInEquilibrium_meshStudy_mesh{Res}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));
    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found!");

        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }

    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
evalSess

Searching for: DropletInEquilibrium_meshStudy_mesh20
found
Searching for: DropletInEquilibrium_meshStudy_mesh40
found
Searching for: DropletInEquilibrium_meshStudy_mesh60
Searching for: DropletInEquilibrium_meshStudy_mesh80


#0: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh20	10/24/2025 4:47:55 PM	ef32ca7d...
#1: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh40	10/24/2025 4:48:06 PM	2791d068...


In [9]:
NUnit.Framework.Assert.AreEqual(4, evalSess.Count, $"Not enough sessions for evaluation.");

Error: NUnit.Framework.AssertionException:   Not enough sessions for evaluation.
  Expected: 4
  But was:  2

   at NUnit.Framework.Assert.ReportFailure(String message)
   at NUnit.Framework.Assert.ReportFailure(ConstraintResult result, String message, Object[] args)
   at NUnit.Framework.Assert.That[TActual](TActual actual, IResolveConstraint expression, String message, Object[] args)
   at NUnit.Framework.Assert.AreEqual(Object expected, Object actual, String message, Object[] args)
   at Submission#9.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

### compute times

In [10]:
bool calcComputeTimes = false;

In [11]:
if (calcComputeTimes) {
    
foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}
}

# Evaluate terminal time step

In [12]:
public static void EvaluateTerminalTimeStep(ISessionInfo sess, double refL2velocity, double refL2velocity_err, double refL2pressure, double refL2pressure_err) {

    string[] nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    Console.WriteLine($"{nameData.Last()}:");

    var terminalStep = sess.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber).Last();
    //Console.WriteLine($"timestep: {terminalStep.TimeStepNumber.MajorNumber} at physical time: {terminalStep.PhysicalTime}");

    // L2 error velocity
    double uL2 = terminalStep.GetField("VelocityX").L2Norm();
    double vL2 = terminalStep.GetField("VelocityY").L2Norm();
    double velL2 = (uL2.Pow2()+vL2.Pow2()).Sqrt();
    Console.WriteLine($"L2-error velocity: {velL2}");
    NUnit.Framework.Assert.AreEqual(refL2velocity, velL2, refL2velocity_err, 
                $"L2-error for velocity does not match reference value {refL2velocity}.");

    // =======================================
    DGField phi = terminalStep.GetField("Phi");
    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
    LevSet.Acc(1.0, phi); 
    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
    LsTrk.UpdateTracker(125.0);

    // get pressure niveau outside droplet
    DGField pressure = terminalStep.GetField("Pressure");
    int order        = 8;
    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order, 1).XQuadSchemeHelper;
    SpeciesId spcBId = LsTrk.SpeciesIdS[1];
    var vqs = SchemeHelper.GetVolumeQuadScheme(spcBId);
    double areaB     = XNSEUtils.GetSpeciesArea(LsTrk, spcBId);
    double p0        = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat, vqs.Compile(LsTrk.GridDat, order),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            for (int i = 0; i < Length; i++) {
                double p0i = ((XDGField)pressure).GetSpeciesShadowField("B").GetMeanValue(i0+i);
                for (int k = 0; k < QR.NoOfNodes; k++) {
                    EvalResult[i,k,0] = p0i;
                }
            }
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for (int i = 0; i < Length; i++) {
                p0 += ResultsOfIntegration[i, 0]/areaB;
            }
        }
    ).Execute();

    // L2 error outside droplet (spceciesB)
    var pOutN = ((XDGField)pressure).GetSpeciesShadowField("B");
    pOutN.AccConstant(-p0);
    double pOutL2 = pOutN.L2Error((ScalarFunction)null, vqs);

    // L2 error inside droplet (spceciesA)
    SpeciesId spcAId = LsTrk.SpeciesIdS[0];
    vqs   = SchemeHelper.GetVolumeQuadScheme(spcAId);
    double sigma = 1.0;
    double r     = 0.25;
    Func<double[], double> refA = (X => sigma/r);
    var pInN     = ((XDGField)pressure).GetSpeciesShadowField("A");
    pInN.AccConstant(-p0);
    double pInL2 = pInN.L2Error(refA.Vectorize(), vqs);

    // L2 error pressure
    double pL2 = (pInL2.Pow2()+pOutL2.Pow2()).Sqrt();  
    Console.WriteLine($"L2-error pressure: {pL2}");
    NUnit.Framework.Assert.AreEqual(refL2pressure, pL2, refL2pressure_err, 
                $"L2-error for pressure does not match reference value {refL2pressure}.");

}

### TABLE 3 - L2-error norms of spurious velocities and Laplace-Young equation for the terminal time step at t = 125

L2-error norms for mesh 20

In [13]:
EvaluateTerminalTimeStep(evalSess.Pick(0), 1.68e-5, 2e-5, 1.03e-2, 0.009)

mesh20:
L2-error velocity: 1.9242132776309145E-06
L2-error pressure: 0.0013806616987715446


L2-error norms for mesh 40

In [14]:
EvaluateTerminalTimeStep(evalSess.Pick(1), 2.95e-7, 9e-8, 5.20e-4, 4e-5)

mesh40:
L2-error velocity: 2.0856934399750723E-07
L2-error pressure: 0.00048677709427256835


L2-error norms for mesh 60

In [15]:
EvaluateTerminalTimeStep(evalSess.Pick(2), 2.60e-7, 2e-10, 5.68e-4, 5e-7)

Error: System.ArgumentOutOfRangeException: Index was out of range. Must be non-negative and less than the size of the collection. (Parameter 'index')
   at System.Collections.Generic.List`1.get_Item(Int32 index)
   at System.Linq.Enumerable.ElementAt[TSource](IEnumerable`1 source, Int32 index)
   at BoSSS.Foundation.IO.IEnumerableExtensions.Pick[T](IEnumerable`1 entities, Int32 index) in C:\BoSSS-experimental\public\src\L4-application\BoSSSpad\IEnumerableExtensions.cs:line 131
   at Submission#15.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

L2-error norms for mesh 80

In [16]:
EvaluateTerminalTimeStep(evalSess.Pick(3), 1.34e-7, 5e-10, 4.15e-4, 4e-7)

Error: System.ArgumentOutOfRangeException: Index was out of range. Must be non-negative and less than the size of the collection. (Parameter 'index')
   at System.Collections.Generic.List`1.get_Item(Int32 index)
   at System.Linq.Enumerable.ElementAt[TSource](IEnumerable`1 source, Int32 index)
   at BoSSS.Foundation.IO.IEnumerableExtensions.Pick[T](IEnumerable`1 entities, Int32 index) in C:\BoSSS-experimental\public\src\L4-application\BoSSSpad\IEnumerableExtensions.cs:line 131
   at Submission#16.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

# Energy evaluation

In [17]:
public static Plot2Ddata GetKineticBulkEnergyPlot(List<ISessionInfo> sessions, string[] meshSizes) {

    Plot2Ddata data = sessions.ReadLogDataValue( "Energy", "kineticEnergy", new string[] { " L2Norm KineticEnergy" });

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "Time";
    plt.Ylabel = "Kinetic bulk energy";
    
    var datGroups = data.dataGroups;
    int lcIdx = 1;
    for (int i = 0; i < datGroups.Count(); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (meshSizes.Contains(nameData[2])) {
            plt.AddDataGroup(nameData[2], datGroups[i].Abscissas, datGroups[i].Values, fmt);
            lcIdx++;
        }
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 25.0;
    plt.ShowLegend = true;

    return plt;

} 

In [18]:
public static Plot2Ddata GetBulkDissipationPlot(List<ISessionInfo> sessions, string[] meshSizes) {

    Plot2Ddata data = sessions.ReadLogDataValue( "Energy", "kineticDissipation", new string[] { " L2Norm KineticDissipationBulk" });

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "Time";
    plt.Ylabel = "Bulk dissipation";
    
    var datGroups = data.dataGroups;
    int lcIdx = 1;
    for (int i = 0; i < datGroups.Count(); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (meshSizes.Contains(nameData[2])) {
            plt.AddDataGroup(nameData[2], datGroups[i].Abscissas, datGroups[i].Values, fmt);
            lcIdx++;
        }
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 25.0;
    plt.ShowLegend = true;
    plt.LegendAlignment = new string[] { "i", "b", "r" };

    return plt;

} 

In [19]:
public static Plot2Ddata GetSurfaceDivergencePlot(List<ISessionInfo> sessions, string[] meshSizes) {

    Plot2Ddata data = sessions.ReadLogDataValue( "Energy", "surfaceDivergence", new string[] { " L2Norm SurfaceDivergence" });

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "Time";
    plt.Ylabel = "Surface divergence";
    
    var datGroups = data.dataGroups;
    int lcIdx = 1;
    for (int i = 0; i < datGroups.Count(); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (meshSizes.Contains(nameData[2])) {
            plt.AddDataGroup(nameData[2], datGroups[i].Abscissas, datGroups[i].Values, fmt);
            lcIdx++;
        }
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 120.0;
    plt.ShowLegend = true;
    plt.LegendAlignment = new string[] { "i", "b", "r" };

    return plt;

} 

## FIGURE 5 

In [20]:
Plot2Ddata[,] Fig5 = new Plot2Ddata[1, 3];

string[] meshStudy = new string[] { "mesh40", "mesh60", "mesh80" };
Fig5[0, 0] = GetKineticBulkEnergyPlot(evalSess, meshStudy);
Fig5[0, 1] = GetBulkDissipationPlot(evalSess, meshStudy);
Fig5[0, 2] = GetSurfaceDivergencePlot(evalSess, meshStudy);

Fig5.ToGnuplot().PlotSVG(xRes:2000,yRes:500)

Processing session: DropletInEquilibrium_meshStudy_mesh20
  Merging data from 1 sessions.
... Done
Processing session: DropletInEquilibrium_meshStudy_mesh40
  Merging data from 1 sessions.
... Done
Build DataSet
Processing session: DropletInEquilibrium_meshStudy_mesh20
  Merging data from 1 sessions.
... Done
Processing session: DropletInEquilibrium_meshStudy_mesh40
  Merging data from 1 sessions.
... Done
Build DataSet
Processing session: DropletInEquilibrium_meshStudy_mesh20
  Merging data from 1 sessions.
valueName surfaceDivergence or alternatives not found ... setting values to zero
... Done
Processing session: DropletInEquilibrium_meshStudy_mesh40
  Merging data from 1 sessions.
valueName surfaceDivergence or alternatives not found ... setting values to zero
... Done
Build DataSet
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to

Gnuplot Error: Warning: empty y range [0:0], adjusting to [-1:1]



<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 1x10 -8 
 
 
 
 
 2x10 -8 
 
 
 
 
 3x10 -8 
 
 
 
 
 4x10 -8 
 
 
 
 
 5x10 -8 
 
 
 
 
 6x10 -8 
 
 
 
 
 0 
 
 
 
 
 5 
 
 
 
 
 10 
 
 
 
 
 15 
 
 
 
 
 20 
 
 
 
 
 25 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Kinetic bulk energy 
 
 
 
 
 Time 
 
 
 
 
 mesh40 
 
 
 
 
 mesh40 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M470.7,62.1 L524.1,62.1 M97.2,301.1 L99.2,207.2 L101.3,132.8 L103.3,93.0 L105.4,89.5 L107.5,103.6
 L109.5,118.9 L111.6,129.2 L113.6,132.6 L115.7,130.4 L117.8,124.8 L119.8,118.0 L121.9,112.5 L123.9,110.9
 L126.0,115.0 L128.1,125.7 L130.1,142.4 L132.2,162.7 L134.2,184.3 L136.3,205.0 L138.4,223.5 L140.4,238.7
 L142.5,251.6 L144.5,263.0 L146.6,274.4 L148.7,287.1 L150.7,301.8 L152.8,318.5 L154.8,336.7 L156.9,355.0
 L159.0,372.6 L161.0,387.8 L163.1,399.9 L165.1,408.1 L167.2,412.6 L169.3,413.2 L171.3,410.7 L173.4,405.6
 L175.4,398.6 L177.5,390.7 L179.6,382.4 L181.6,374.6 L183.7,367.7 L185.7,362.4 L187.8,358.7 L189.9,357.0
 L191.9,357.2 L194.0,359.1 L196.0,362.5 L198.1,367.0 L200.2,372.4 L202.2,378.3 L204.3,384.5 L206.3,390.6
 L208.4,396.6 L210.5,402.3 L212.5,407.6 L214.6,412.4 L216.6,416.7 L218.7,420.2 L220.8,423.1 L222.8,425.1
 L224.9,426.2 L226.9,426.4 L229.0,425.7 L231.1,424.2 L233.1,422.0 L235.2,419.2 L237.2,416.0 L239.3,412.6
 L241.4,409.3 L243.4,406.2 L245.5,403.6 L247.5,401.6 L249.6,400.3 L251.7,399.7 L253.7,400.0 L255.8,401.0
 L257.8,402.6 L259.9,404.8 L262.0,407.5 L264.0,410.4 L266.1,413.4 L268.1,416.4 L270.2,419.2 L272.3,421.8
 L274.3,424.1 L276.4,426.0 L278.4,427.4 L280.5,428.4 L282.6,429.0 L284.6,429.1 L286.7,428.9 L288.7,428.4
 L290.8,427.6 L292.9,426.5 L294.9,425.4 L297.0,424.2 L299.0,423.0 L301.1,421.9 L303.2,420.9 L305.2,420.1
 L307.3,419.6 L309.3,419.3 L311.4,419.3 L313.5,419.6 L315.5,420.2 L317.6,421.0 L319.6,422.0 L321.7,423.1
 L323.8,424.3 L325.8,425.6 L327.9,426.8 L329.9,428.0 L332.0,429.0 L334.1,429.9 L336.1,430.6 L338.2,431.1
 L340.2,431.4 L342.3,431.5 L344.4,431.4 L346.4,431.1 L348.5,430.7 L350.5,430.3 L352.6,429.7 L354.7,429.1
 L356.7,428.5 L358.8,428.0 L360.8,427.5 L362.9,427.1 L365.0,426.8 L367.0,426.6 L369.1,426.5 L371.1,426.6
 L373.2,426.8 L375.3,427.1 L377.3,427.5 L379.4,427.9 L381.4,428.4 L383.5,428.9 L385.6,429.4 L387.6,429.9
 L389.7,430.4 L391.7,430.8 L393.8,431.1 L395.9,431.4 L397.9,431.5 L400.0,431.6 L402.0,431.6 L404.1,431.5
 L406.2,431.3 L408.2,431.1 L410.3,430.9 L412.3,430.6 L414.4,430.4 L416.5,430.1 L418.5,429.9 L420.6,429.7
 L422.6,429.5 L424.7,429.4 L426.8,429.4 L428.8,429.4 L430.9,429.4 L432.9,429.6 L435.0,429.7 L437.1,429.9
 L439.1,430.1 L441.2,430.4 L443.2,430.6 L445.3,430.8 L447.4,431.1 L449.4,431.3 L451.5,431.4 L453.5,431.6
 L455.6,431.7 L457.7,431.8 L459.7,431.8 L461.8,431.8 L463.8,431.8 L465.9,431.7 L468.0,431.7 L470.0,431.6
 L472.1,431.5 L474.1,431.5 L476.2,431.4 L478.3,431.3 L480.3,431.3 L482.4,431.3 L484.4,431.3 L486.5,431.3
 L488.6,431.4 L490.6,431.5 L492.7,431.6 L494.7,431.7 L496.8,431.8 L498.9,431.9 L500.9,432.1 L503.0,432.2
 L505.0,432.3 L507.1,432.4 L509.2,432.5 L511.2,432.6 L513.3,432.6 L515.3,432.6 L517.4,432.7 L519.5,432.6
 L521.5,432.6 L523.6,432.6 L525.6,432.5 L527.7,432.5 L529.8,432.4 L531.8,432.4 L533.9,432.3 L535.9,432.2
 L538.0,432.2 L540.1,432.1 L542.1,432.1 L544.2,432.1 L546.2,432.0 L548.3,432.0 L550.4,432.0 L552.4,432.0
 L554.5,432.1 L556.5,432.1 L558.6,432.1 L560.7,432.1 L562.7,432.2 L564.8,432.2 L566.8,432.2 L568.9,432.2
 L571.0,432.2 L573.0,432.2 L575.1,432.2 L577.1,432.2 L579.2,432.2 L581.3,432.2 L583.3,432.2 L585.4,432.2
 L587.4,432.2 L589.5,432.2 L591.6,432.2 L593.6,432.2 L595.7,432.2 L597.7,432.2 L599.8,432.2 L601.9,432.2
 L603.9,432.2 L606.0,432.3 L608.0

In [21]:
public static void EvaluateAtTime(Plot2Ddata pltDat, double time, double[] lowerThrshldValues, double[] upperThrshldValues) {

    int numGrps = pltDat.dataGroups.Count();
    if (lowerThrshldValues.Length != numGrps || upperThrshldValues.Length != numGrps) {
        throw new ArgumentException("Threshold value arrays must match number of data groups.");
    }

    for (int i = 0; i < numGrps; i++) {
        var grp = pltDat.dataGroups[i];
        // find index at given time
        int timeIdx = -1;
        for (int tI = 0; tI < grp.Abscissas.Length; tI++) {
            if (grp.Abscissas[tI] >= time) {
                // check if this is the closest time point
                if (tI > 0) {
                    double dtPrev = Math.Abs(grp.Abscissas[tI-1] - time);
                    double dtCurr = Math.Abs(grp.Abscissas[tI] - time);
                    if (dtPrev < dtCurr) {
                        timeIdx = tI-1;
                    } else {
                        timeIdx = tI;
                    }
                } else {
                    timeIdx = tI;
                }
                break;
            }
        }
        if (timeIdx == -1) {
            throw new ArgumentException($"Given time {time} exceeds data range.");
        }

        double valAtTime = grp.Values[timeIdx];
        Console.WriteLine($"Data group: {grp.Name}, value at time {time}: {valAtTime}");

        NUnit.Framework.Assert.IsTrue(valAtTime >= lowerThrshldValues[i], 
            $"Value {valAtTime} of data group {grp.Name} at time {time} is below lower threshold {lowerThrshldValues[i]}.");

        NUnit.Framework.Assert.IsTrue(valAtTime <= upperThrshldValues[i], 
            $"Value {valAtTime} of data group {grp.Name} at time {time} is above upper threshold {upperThrshldValues[i]}.");
    }
}

In [22]:
EvaluateAtTime(Fig5[0,0], 25.0, new double[] { 0.0, 0.0, 0.0 }, new double[] { 1.0e-9, 1.0e-9, 1.0e-9 } )

Error: System.ArgumentException: Threshold value arrays must match number of data groups.
   at Submission#21.EvaluateAtTime(Plot2Ddata pltDat, Double time, Double[] lowerThrshldValues, Double[] upperThrshldValues)
   at Submission#22.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [23]:
EvaluateAtTime(Fig5[0,1], 25.0, new double[] { -1.0e-8, -1.0e-8, -1.0e-8 }, new double[] { 0.0, 0.0, 0.0 } )

Error: System.ArgumentException: Threshold value arrays must match number of data groups.
   at Submission#21.EvaluateAtTime(Plot2Ddata pltDat, Double time, Double[] lowerThrshldValues, Double[] upperThrshldValues)
   at Submission#23.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [24]:
EvaluateAtTime(Fig5[0,2], 120.0, new double[] { -2.0e-5, -2.0e-5, -1.0e-5 }, new double[] { 2.0e-5, 2.0e-5, 1.0e-5 } )

Error: System.ArgumentException: Threshold value arrays must match number of data groups.
   at Submission#21.EvaluateAtTime(Plot2Ddata pltDat, Double time, Double[] lowerThrshldValues, Double[] upperThrshldValues)
   at Submission#24.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)